# Week 3, day 3 (afternoon) — Worksheet 03 SOLUTIONS: look at the file first

Executed in the lab image (pandas 3.0.5) against the real lab CSVs. Every
quoted number is what it actually printed.

Questions 3 and 5 are the ones to re-read. The filename makes a claim about the
data that the data contradicts, and the lab uses that filename as its lineage
tag.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 03 — Look at the file first. Run this once.
import pandas as pd

PRODUCTS = "data/products_2013_01_01.csv"
SALES = "data/sales_2013_01_01.csv"

print("these are the two files snowflake-scripts/3_Stage_tables.sql loads")
print("products:", PRODUCTS)
print("sales:   ", SALES)

PART A — bytes before DataFrames

### Question 1

Before using pandas at all, print the first three raw lines of the sales file with plain `open()`. Look at how the text fields are quoted.
> **NOTE:** the `FILE_FORMAT` in the lab's COPY statement sets `FIELD_OPTIONALLY_ENCLOSED_BY = '"'`. Count the quote characters actually present.

In [ ]:
with open(SALES) as fh:
    for i, line in enumerate(fh):
        if i >= 3:
            break
        print("line %d: %s" % (i, line.rstrip()[:110]))

Every text field carries **three** quote characters on each side, not one:

```
247136,1021964,8106,3/7/2010,19,"""Medium""",0.9,30.98,49.61,0,72.8,-41.82,"""Regular Air""",17.08
```

`FIELD_OPTIONALLY_ENCLOSED_BY = '"'` strips **one** pair. The other pair is data.
The value that lands in the table is `"Medium"`, quotes included — see Q5.

Also visible on line 1: `TRANS_DT` is `3/7/2010`, single-digit month and day. That
is why the COPY needs `DATE_FORMAT = 'MM/DD/YYYY'` rather than Snowflake's default
ISO parse.

### Question 2

Load both files with a plain `read_csv` and print the shape and dtypes of each.

In [ ]:
products = pd.read_csv(PRODUCTS)
sales = pd.read_csv(SALES)
print("products:", products.shape)
print("sales:   ", sales.shape)
print()
print(sales.dtypes.to_string())

```
products: (1215, 11)
sales:    (100000, 14)
```

The dtypes are worth noting for later:

```
TRANS_DT           str       <- a date stored as text
TRANS_TIME       int64       <- an hour stored as a number
PRIORITY           str
SALES_QTY      float64       <- a quantity, but fractional
SHIPMODE           str
```

`SALES_QTY` being a float is not a parsing accident — the first row really does
sell `0.9` of something.

### Question 3

The sales file is called `sales_2013_01_01.csv`. Parse `TRANS_DT` with `format="%m/%d/%Y"` and print the earliest date, the latest date, the number of distinct dates, and how many rows are actually dated 2013-01-01.
> **NOTE:** the lab's COPY statement stores this filename in `BATCH_ID`, described in the script as tracking the source file 'for lineage'.

In [ ]:
sales = pd.read_csv(SALES)
d = pd.to_datetime(sales["TRANS_DT"], format="%m/%d/%Y")
print("filename says:      2013_01_01")
print("earliest TRANS_DT:", d.min().date())
print("latest TRANS_DT:  ", d.max().date())
print("distinct dates:   ", d.nunique())
print("rows dated 2013-01-01:", int((d == "2013-01-01").sum()))
print()
print(d.dt.year.value_counts().sort_index().to_string())

The filename is wrong about its own contents.

```
filename says:      2013_01_01
earliest TRANS_DT: 2009-01-01
latest TRANS_DT:   2012-12-30
distinct dates:    1411
rows dated 2013-01-01: 0
```

Four years of history, spread over 1,411 distinct days:

```
2009    25761
2010    25569
2011    23679
2012    24991
```

**Not one row falls on the date in the filename.** The lab's COPY statement does
this:

```sql
METADATA$FILENAME AS BATCH_ID
```

and the script comments describe it as tracking the source file for lineage. As
lineage that is fine — it faithfully records which file a row came from. The trap
is reading `BATCH_ID` as a date. Anyone who filters `WHERE BATCH_ID LIKE '%2013_01_01%'`
expecting New Year's Day 2013 gets four years of sales; anyone who partitions by it
gets one partition holding everything.

Filenames are labels somebody typed. `TRANS_DT` is the data.

PART B — the columns the loader will not check

### Question 4

Print the sales file's column names. Compare them with the columns `STG_Sales` declares in `snowflake-scripts/3_Stage_tables.sql` and find the one that does not match.

In [ ]:
sales = pd.read_csv(SALES)
csv_cols = list(sales.columns)
stg_cols = ["TRANS_ID", "PROD_KEY", "STORE_KEY", "TRANS_DT", "TRANS_TIME",
            "PRIORITY", "SALES_QTY", "SALES_PRICE", "SALES_AMT",
            "DISCOUNT", "SALES_COST", "SALES_MGRN", "SHIP_MODE", "SHIP_COST"]
print("in the CSV but not the table:", [c for c in csv_cols if c not in stg_cols])
print("in the table but not the CSV:", [c for c in stg_cols if c not in csv_cols])
print()
for i, (a, b) in enumerate(zip(csv_cols, stg_cols), 1):
    if a != b:
        print("  position %d: CSV says %r, table says %r" % (i, a, b))

One column name differs, at position 13:

```
in the CSV but not the table: ['SHIPMODE']
in the table but not the CSV: ['SHIP_MODE']

  position 13: CSV says 'SHIPMODE', table says 'SHIP_MODE'
```

The header says `SHIPMODE`; `STG_Sales` declares `SHIP_MODE`. Both are position 13,
and the lab's COPY selects `$1, $2, ... $13, $14` — by **position**. The header row
is skipped (`SKIP_HEADER = 1`) and never compared to anything, so the load succeeds
and the data lands in the right column.

That is the load working correctly. It is also the load being unable to warn you:
had two columns been swapped in the file, position-based loading would have put
`SHIP_COST` into `SHIP_MODE` just as silently. Q10 is what happens when something
matches by name instead.

### Question 5

Print the distinct values of `PRIORITY` and `SHIPMODE` using `repr()`. Say what a `WHERE PRIORITY = 'High'` would return after this file is loaded as-is.
> **NOTE:** compare with Q1. The `FIELD_OPTIONALLY_ENCLOSED_BY` setting strips one layer of quoting.

In [ ]:
sales = pd.read_csv(SALES)
print("PRIORITY:")
for v in sorted(sales["PRIORITY"].unique()):
    print("   ", repr(v))
print()
print("SHIPMODE:")
for v in sorted(sales["SHIPMODE"].unique()):
    print("   ", repr(v))
print()
print("rows where PRIORITY == 'High':", int((sales["PRIORITY"] == "High").sum()))
print("rows where PRIORITY == '\"High\"':", int((sales["PRIORITY"] == '"High"').sum()))

Every value arrives wearing quotes:

```
PRIORITY:
    '"Critical"'
    '"High"'
    '"Low"'
    '"Medium"'
    '"Not Specified"'

SHIPMODE:
    '"Delivery Truck"'
    '"Express Air"'
    '"Regular Air"'
```

So:

```
rows where PRIORITY == 'High': 0
rows where PRIORITY == '"High"': 21117
```

`WHERE PRIORITY = 'High'` returns **nothing**. Not an error, not a warning — an
empty result set, which reads exactly like "there were no high-priority orders."
21,117 of them are sitting in the table.

This is the failure mode worth fearing: the load reported success, the row count
validated, the column is populated, and the query is silently wrong. Fix it at load
time in the transformational SELECT, not in every query afterwards:

```sql
TRIM($6, '"') AS PRIORITY
```

Values a `GROUP BY` will show you are cheap to audit. Do it once, on arrival.

PART C — keys that are not keys

### Question 6

`TRANS_ID` looks like a transaction identifier. Count the rows, the distinct `TRANS_ID` values, and how many rows are duplicates of an earlier one. Then find what makes a row unique: test `TRANS_ID` with `PROD_KEY`, and with `STORE_KEY`, using `drop_duplicates`.
> **NOTE:** do not stop at "TRANS_ID is not unique". A repeated key is a clue about the grain of the table — what one row means.

In [ ]:
sales = pd.read_csv(SALES)
print("rows:             ", len(sales))
print("distinct TRANS_ID:", sales["TRANS_ID"].nunique())
print("duplicated rows:  ", int(sales["TRANS_ID"].duplicated().sum()))
print()
for combo in (["TRANS_ID"], ["TRANS_ID", "PROD_KEY"], ["TRANS_ID", "STORE_KEY"]):
    n = len(sales.drop_duplicates(combo))
    print("  %-28s -> %6d distinct, %6d duplicated"
          % (" + ".join(combo), n, len(sales) - n))
print()
one = sales[sales["TRANS_ID"] == 247136]
print("every row for TRANS_ID 247136:")
print(one[["TRANS_ID", "STORE_KEY", "PROD_KEY", "TRANS_DT", "SALES_AMT"]].to_string(index=False))

`TRANS_ID` is not unique, and not by a small margin:

```
rows:              100000
distinct TRANS_ID: 7916
duplicated rows:   92084
```

The obvious guess is line items — one order, several products. The composite test
says otherwise:

```
  TRANS_ID                     ->   7916 distinct,  92084 duplicated
  TRANS_ID + PROD_KEY          ->   7916 distinct,  92084 duplicated
  TRANS_ID + STORE_KEY         -> 100000 distinct,      0 duplicated
```

Adding `PROD_KEY` changes **nothing** — so every row sharing a `TRANS_ID` shares its
product too. Adding `STORE_KEY` makes every row unique. One transaction row for one
product across many stores:

```
 TRANS_ID  STORE_KEY  PROD_KEY TRANS_DT  SALES_AMT
   247136       8106   1021964 3/7/2010      49.61
   247136       8107   1021964 3/7/2010      44.65
   247136       9001   1021964 3/7/2010      34.73
   ...
   247136       9013   1021964 3/7/2010      49.61
```

Thirteen rows, one product, one date, thirteen different stores, and thirteen
different amounts. The grain of this table is **(TRANS_ID, STORE_KEY)** — sales of
one product, on one day, per store.

Two things follow. `COUNT(DISTINCT TRANS_ID)` is 7,916 and `COUNT(*)` is 100,000, and
calling either one "transactions" is wrong — the first counts product-days, the
second counts product-store-days. And `TRANS_ID` cannot be a primary key, so if you
were planning to deduplicate on it or merge on it, you would have collapsed twelve
stores into one.

The general move is the one this question makes: when a key repeats, do not just
record that it repeats. Find the column that completes it. That column tells you
what a row actually means.

### Question 7

Do the same for `PROD_KEY` in the products file — the product *master*, where one row should be one product.

In [ ]:
products = pd.read_csv(PRODUCTS)
print("rows:             ", len(products))
print("distinct PROD_KEY:", products["PROD_KEY"].nunique())
print("duplicated:       ", int(products["PROD_KEY"].duplicated().sum()))
print()
dupes = products[products["PROD_KEY"].duplicated(keep=False)].sort_values("PROD_KEY")
print(dupes[["PROD_KEY", "PROD_NAME", "BRAND_NAME", "CATEGORY_NAME"]].to_string(index=False))

The product master has duplicates too — three of them:

```
rows:              1215
distinct PROD_KEY: 1212
duplicated:        3
```

and they are not all the same kind of duplicate:

```
 PROD_KEY      PROD_NAME BRAND_NAME CATEGORY_NAME
    72479  Product-72479    brand-1    category-4
    72479  Product-72479    brand-9    category-5
   481924 Product-481924   brand-14    category-3
   481924 Product-481924   brand-19    category-4
   861428 Product-861428   brand-18    category-5
   861428 Product-861428   brand-18    category-5
```

`861428` appears twice **identically** — a harmless double-insert, `DISTINCT` removes it.

`72479` and `481924` are the dangerous ones. Same key, **different brand, different
category**. There is no correct answer to "what category is product 72479 in" — the
file asserts both. `DISTINCT` will not save you; it keeps both rows because they
genuinely differ.

Now join sales to products on `PROD_KEY`. Every sale of 72479 matches two product
rows, so it appears **twice** in the result, and any revenue total is inflated. Three
bad rows in a 1,215-row dimension quietly corrupts a 100,000-row fact join.

This is why loading a dimension table means picking a rule — latest wins, source
priority, or reject and escalate — rather than trusting the key.

### Question 8

Check whether every `PROD_KEY` in sales exists in products. Print the counts and the number of sales rows referencing a product that is not there.
> **NOTE:** this is the check a `COPY INTO` cannot do for you — the two files load into two independent tables.

In [ ]:
products = pd.read_csv(PRODUCTS)
sales = pd.read_csv(SALES)
prod_keys = set(products["PROD_KEY"])
sale_keys = set(sales["PROD_KEY"])
print("distinct PROD_KEY in products:", len(prod_keys))
print("distinct PROD_KEY in sales:   ", len(sale_keys))
print("in sales but not products:    ", len(sale_keys - prod_keys))
print("in products but never sold:   ", len(prod_keys - sale_keys))
print()
orphans = sales["PROD_KEY"].isin(sale_keys - prod_keys).sum()
print("sales rows with no matching product:", int(orphans))

This one is clean:

```
distinct PROD_KEY in products: 1212
distinct PROD_KEY in sales:    1212
in sales but not products:     0
in products but never sold:    0

sales rows with no matching product: 0
```

Perfect referential integrity in both directions. Every product sold exists; every
product listed sold at least once.

That is a real property of this dataset and worth confirming — but notice what
confirmed it. Not the load. `COPY INTO` filled `STG_Products` and `STG_Sales`
independently; neither statement can see the other's table, and Snowflake does not
enforce foreign keys even when you declare them. Had 400 sales rows pointed at
products that do not exist, both COPYs would still have reported success and the
row counts in `4_Validate_stage_tables.sql` would still have matched.

Cross-table checks are checks you write, after the load, deliberately.

### Question 9

The lab loads `SALES_QTY`, `SALES_PRICE` and `SALES_AMT` as three independent columns. Check whether `SALES_AMT` equals `SALES_QTY * SALES_PRICE`, rounded to 2dp, and print how many rows agree. Show two rows that do not.

In [ ]:
sales = pd.read_csv(SALES)
calc = (sales["SALES_QTY"] * sales["SALES_PRICE"]).round(2)
agree = int((calc == sales["SALES_AMT"].round(2)).sum())
print("rows where QTY * PRICE == SALES_AMT:", agree, "of", len(sales))
print()
bad = sales[calc != sales["SALES_AMT"].round(2)].head(2)
for _, r in bad.iterrows():
    print("  qty %.2f x price %.2f = %.2f, but SALES_AMT is %.2f (discount %.2f)"
          % (r["SALES_QTY"], r["SALES_PRICE"],
             r["SALES_QTY"] * r["SALES_PRICE"], r["SALES_AMT"], r["DISCOUNT"]))

The three money columns do not agree with each other:

```
rows where QTY * PRICE == SALES_AMT: 33 of 100000
```

33 rows out of 100,000 — 0.03%. Effectively, `SALES_AMT` is never the product of the
other two:

```
  qty 0.90 x price 30.98 = 27.88, but SALES_AMT is 49.61 (discount 0.00)
  qty 30.10 x price 54.96 = 1654.30, but SALES_AMT is 1724.52 (discount 0.03)
```

And discount does not explain it. Row 1 has a discount of `0.00` and `SALES_AMT` is
*higher* than qty x price, not lower. Whatever produced these columns, it was not
this arithmetic.

The practical consequence: `SUM(SALES_AMT)` and `SUM(SALES_QTY * SALES_PRICE)` are
two different revenue numbers from the same table, and both look defensible in a
dashboard. Someone has to decide which one the business means — and that decision
belongs in the model, written down, not rediscovered by whoever writes the next query.

You cannot derive that answer from the file. You can only detect, as here, that the
file will not answer it for you.

### Question 10

Finally, read the sales file the way the staging table names its columns — `pd.read_csv(SALES, usecols=["TRANS_ID", "SHIP_MODE"])`. **This is supposed to fail.** Read the exception and connect it to Q4.
> **NOTE:** the lab's `COPY INTO` loads by POSITION (`$1, $2, ...`), not by name, so it never notices this. Reading by name does.

In [ ]:
print("columns actually in the file:", [c for c in pd.read_csv(SALES, nrows=0).columns if "SHIP" in c])
print(pd.read_csv(SALES, usecols=["TRANS_ID", "SHIP_MODE"]))

It raises:

```
ValueError: Usecols do not match columns, columns expected but not found: ['SHIP_MODE']
```

because the file's header actually reads:

```
columns actually in the file: ['SHIPMODE', 'SHIP_COST']
```

Same mismatch as Q4 — and this time it stopped the read.

The contrast is the whole point of this worksheet. `COPY INTO` selects `$13` and gets
the thirteenth field, whatever it is called; the mismatch is invisible and the load
succeeds. `read_csv(usecols=[...])` asks for a column *by name*, does not find it, and
refuses.

Neither behaviour is wrong. Position-based loading is what makes `COPY INTO` fast and
tolerant of header noise. But it means the loader's definition of success — "the bytes
arrived in the right slots" — is narrower than yours. Everything this worksheet found
happened **after** a completely successful load:

- a filename claiming a date the data contradicts (Q3)
- a header name the table does not use (Q4)
- quotes that make `WHERE PRIORITY = 'High'` return zero rows (Q5)
- a "transaction id" repeated 92,084 times (Q6)
- two products with contradictory categories under one key (Q7)
- three money columns that disagree with each other (Q9)

`4_Validate_stage_tables.sql` counts rows. Rows arriving is the easy part.